# DM-SR Study — Reference + COLMAP pack (CPU only)

Este notebook **não usa GPU** e não executa treino, avaliação ou meshing. Ele baixa o ZIP oficial DM-SR, valida o SHA-256 congelado e extrai em uma única execução: a malha ground truth `study.ply`, as 300 imagens RGB de treino + `train/transforms.json` necessárias ao baseline geométrico COLMAP, e quatro imagens RGB de teste + `test/transforms.json` para figuras qualitativas.

No Kaggle: deixe **Accelerator = None**, habilite **Internet** e use **Save Version → Save & Run All**. O único output necessário será `dmsr-study-reference-pack.zip`.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, zipfile

URL = 'https://www.dropbox.com/s/1k75m38vahizbp9/dmsr.zip?dl=1'
DATASET_SHA256 = '146ddd88d34cf8efc3357737871325f2ad1fe7584f7e20fc8ec75f5d3baed602'
GT_SHA256 = '8de1c71689452af874d661b610873a036fddf979b89c86704e5537561c1452fc'
ROOT = Path('/kaggle/temp/dmsr-reference-pack')
OUT = Path('/kaggle/working/dmsr-study-reference-pack')
ARCHIVE = ROOT / 'dmsr.zip'
ROOT.mkdir(parents=True, exist_ok=True)
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

subprocess.run([
    'curl','--fail','--location','--retry','3','--connect-timeout','30','--max-time','1800',
    '--output',str(ARCHIVE),URL
], check=True)
assert zipfile.is_zipfile(ARCHIVE), 'Download oficial não é ZIP'
actual = sha256(ARCHIVE)
assert actual == DATASET_SHA256, f'SHA do dataset divergente: {actual}'

with zipfile.ZipFile(ARCHIVE) as z:
    names = set(z.namelist())
    fixed = [
        'dmsr/study/study.ply',
        'dmsr/study/train/transforms.json',
        'dmsr/study/test/transforms.json',
    ]
    train_rgbs = sorted(n for n in names if n.startswith('dmsr/study/train/rgbs/') and n.endswith('.png'))
    test_views = [0, 33, 66, 99]
    test_rgbs = [f'dmsr/study/test/rgbs/r_{i:04d}.png' for i in test_views]
    missing = [m for m in fixed + test_rgbs if m not in names]
    assert not missing, f'Arquivos ausentes no ZIP: {missing}'
    assert len(train_rgbs) == 300, f'Esperadas 300 imagens RGB de treino, encontradas {len(train_rgbs)}'
    members = fixed + train_rgbs + test_rgbs
    for member in members:
        rel = Path(member).relative_to('dmsr/study')
        target = OUT / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(member) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

gt = OUT / 'study.ply'
gt_sha = sha256(gt)
assert gt_sha == GT_SHA256, f'SHA do study.ply divergente: {gt_sha}'
manifest = {
    'dataset_url': URL,
    'dataset_sha256': DATASET_SHA256,
    'study_ply_sha256': gt_sha,
    'study_ply_bytes': gt.stat().st_size,
    'train_rgb_count': len(train_rgbs),
    'test_rgb_views': test_views,
    'purpose': ['geometry_ground_truth', 'qualitative_rgb', 'colmap_geometry_baseline'],
}
(OUT/'reference_manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')

archive_out = shutil.make_archive('/kaggle/working/dmsr-study-reference-pack', 'zip', OUT)
ARCHIVE.unlink(missing_ok=True)
print(json.dumps(manifest, indent=2))
print('RESULTS:', archive_out)
